# Práctica N°03: Simulación de eventos discretos

# 5.4. Simulación de eventos discretos en SimPy

Instalación del framework SimPy y desarrollo paso a paso de los tutoriales y ejemplos indicados en la guía.

## Instalación de SimPy

In [1]:
!pip install simpy

In [2]:
import itertools
import random
from typing import Dict, List, NamedTuple, Optional

import simpy

print(simpy.__version__)

4.1.2


## 5.4.1. Event discrete simulation with SimPy (Stefan Scherfke) y notebook *Discrete-event simulation with SimPy.ipynb*

Presentación: https://www.youtube.com/watch?v=Bk91DoAEcjY · Diapositivas: https://stefan.sofa-rockers.org/downloads/simpy-ep14.pdf

Se desarrollan paso a paso los ejemplos de la presentación usando el notebook de la guía: funciones generadoras, `Environment`, eventos *timeout*, procesos como eventos, interrupciones, eventos de condición y el ejemplo del asistente a la conferencia (con y sin buffet).

# Funciones generadoras para procesos de modelado

Ejemplo 1

In [3]:
def generator(x):
  y = yield x + 1
  return y + 1

In [4]:
g = generator(3)

In [5]:
next(g)

4

Ejemplo 2

In [6]:
def double_inputs():
  while True:
    x = yield
    yield x * 2

In [7]:
gen = double_inputs()

In [8]:
next(gen)

In [9]:
gen.send(5)

10

In [10]:
next(gen)

In [11]:
gen.send(6)

12

In [12]:
next(gen)

In [13]:
gen.send(94.3)

188.6

# Ejemplo

In [14]:
def clock(env, name, tick):
  while True:
    print(name, env.now)
    yield env.timeout(tick)

In [15]:
env = simpy.Environment()

In [16]:
env.process(clock(env, 'rápido', 0.5))

<Process(clock) object at 0x7fc04cb9a410>

In [17]:
env.process(clock(env, 'lento', 1))

<Process(clock) object at 0x7fc04cb27710>

In [18]:
env.run(until=5)

rápido 0
lento 0
rápido 0.5
lento 1
rápido 1.0
rápido 1.5
lento 2
rápido 2.0
rápido 2.5
lento 3
rápido 3.0
rápido 3.5
lento 4
rápido 4.0
rápido 4.5


In [19]:
env = simpy.Environment()

In [20]:
env.process(clock(env, 'rápido', 0.5))

<Process(clock) object at 0x7fc04cba0450>

In [21]:
env.process(clock(env, 'lento', 1))

<Process(clock) object at 0x7fc04cba0a90>

In [22]:
env.process(clock(env, 'muy rápido', 0.01))

<Process(clock) object at 0x7fc04cb9fe10>

In [23]:
env.run(until=5)

rápido 0
lento 0
muy rápido 0
muy rápido 0.01
muy rápido 0.02
muy rápido 0.03
muy rápido 0.04
muy rápido 0.05
muy rápido 0.060000000000000005
muy rápido 0.07
muy rápido 0.08
muy rápido 0.09
muy rápido 0.09999999999999999
muy rápido 0.10999999999999999
muy rápido 0.11999999999999998
muy rápido 0.12999999999999998
muy rápido 0.13999999999999999
muy rápido 0.15
muy rápido 0.16
muy rápido 0.17
muy rápido 0.18000000000000002
muy rápido 0.19000000000000003
muy rápido 0.20000000000000004
muy rápido 0.21000000000000005
muy rápido 0.22000000000000006
muy rápido 0.23000000000000007
muy rápido 0.24000000000000007
muy rápido 0.25000000000000006
muy rápido 0.26000000000000006
muy rápido 0.2700000000000001
muy rápido 0.2800000000000001
muy rápido 0.2900000000000001
muy rápido 0.3000000000000001
muy rápido 0.3100000000000001
muy rápido 0.3200000000000001
muy rápido 0.3300000000000001
muy rápido 0.34000000000000014
muy rápido 0.35000000000000014
muy rápido 0.36000000000000015
muy rápido 0.370000000000

# **Environment**<br>
simular “tan rápido como sea posible”

# **RealtimeEnvironment**<br>
sincronizado con el tiempo del reloj de pared

# Los eventos de tiempo de espera dejan pasar el tiempo

In [24]:
def speaker(env, start):
  print('t: '+str(env.now))
  until_start = start - env.now
  yield env.timeout(until_start)
  print('t: '+str(env.now))
  yield env.timeout(30)
  print('t: '+str(env.now))

env = simpy.Environment()
env.process(speaker(env, 5))
env.run(100)

t: 0
t: 5
t: 35


# Los procesos son eventos, también

In [25]:
def speaker(env):
  yield env.timeout(30)
  return 'orador: '+ str(env.now) + ' - finaliza charla'

def moderator(env):
  for i in range(5):
    print('moderador: ' + str(env.now))
    val = yield env.process(speaker(env))
    print(val)

env = simpy.Environment()
env.process(moderator(env))
env.run(until=151)

moderador: 0
orador: 30 - finaliza charla
moderador: 30
orador: 60 - finaliza charla
moderador: 60
orador: 90 - finaliza charla
moderador: 90
orador: 120 - finaliza charla
moderador: 120
orador: 150 - finaliza charla


# Interrupciones asincrónicas

**Nota.** Como explica Stefan Scherfke en la presentación, este ejemplo tiene un problema: si el orador habla menos de 30 min, su proceso ya terminó y `interrupt()` lanza `RuntimeError`. Se fija una semilla (`random.seed(11)`) con la que los tres oradores superan los 30 min para que la celda corra; el problema se resuelve en la siguiente sección con eventos de condición.

In [26]:
random.seed(11)

def speaker(env):
    try:
        tspeaker = random.randint(25, 35)
        yield env.timeout(tspeaker)
        print('orador: '+ str(env.now) + ' - finaliza charla')
    except simpy.Interrupt as interrupt:
        print(interrupt.cause)

def moderator(env):
    for i in range(3):
        print('moderador: ' + str(env.now))
        speaker_proc = env.process(speaker(env))
        yield env.timeout(30)
        speaker_proc.interrupt('moderador: No queda tiempo.')

env = simpy.Environment()
env.process(moderator(env))
env.run(until=120)

moderador: 0
moderador: 30
moderador: No queda tiempo.
moderador: 60
moderador: No queda tiempo.
moderador: No queda tiempo.


# Eventos de condición

In [27]:
def speaker(env):
    try:
        ts = random.randint(25, 35)
        print('orador inicia en: '+str(env.now))
        print('duración de la charla: '+str(ts))
        yield env.timeout(ts)
        print('orador finaliza en: '+str(env.now))
    except simpy.Interrupt as interrupt:
        print(interrupt.cause)

def moderator(env):
    for i in range(10):
        print("El moderador deja que el orador número {} para comenzar a hablar".format(i))
        speaker_proc = env.process(speaker(env))

        print("Tiempo actual: {}".format(env.now))
        time_limit = env.timeout(30)
        print('Tiempo muerto (timeout) creado')
        results = yield speaker_proc | time_limit
        print("El moderador verifica si el orador se pasó del tiempo limite o no")
        print("Límite de tiempo superado: {}".format(speaker_proc not in results))

        if speaker_proc not in results:

            print("Tiempo actual: {}".format(env.now))
            print(speaker_proc.is_alive)
            speaker_proc.interrupt()
            print("El moderador detiene la charla en: {}".format(env.now))
        print()
        print()

env = simpy.Environment()
env.process(moderator(env))
env.run(until=500)

El moderador deja que el orador número 0 para comenzar a hablar
Tiempo actual: 0
Tiempo muerto (timeout) creado
orador inicia en: 0
duración de la charla: 32
El moderador verifica si el orador se pasó del tiempo limite o no
Límite de tiempo superado: True
Tiempo actual: 30
True
El moderador detiene la charla en: 30


El moderador deja que el orador número 1 para comenzar a hablar
Tiempo actual: 30
Tiempo muerto (timeout) creado
None
orador inicia en: 30
duración de la charla: 33
El moderador verifica si el orador se pasó del tiempo limite o no
Límite de tiempo superado: True
Tiempo actual: 60
True
El moderador detiene la charla en: 60


El moderador deja que el orador número 2 para comenzar a hablar
Tiempo actual: 60
Tiempo muerto (timeout) creado
None
orador inicia en: 60
duración de la charla: 34
El moderador verifica si el orador se pasó del tiempo limite o no
Límite de tiempo superado: True
Tiempo actual: 90
True
El moderador detiene la charla en: 90


El moderador deja que el orad

# **Ejemplo**: **Asistente a la conferencia**

In [28]:
TALKS_PER_SESSION = 3
TALK_LENGTH = 20
BREAK_LENGTH = 15

In [29]:
def attendee(env, name, knowledge=0, hunger=0):
  while True:
    for i in range(TALKS_PER_SESSION):
      knowledge += random.randint(0, 3) / (1 + hunger)
      hunger += random.randint(1, 4)
      yield env.timeout(TALK_LENGTH)
    print('El asistente %s terminó las charlas con un conocimiento %.2f y hambre '
    '%.2f.' % (name, knowledge, hunger))

    food = random.randint(3, 12)
    hunger -= min(food, hunger)

    yield env.timeout(BREAK_LENGTH)
    print('El asistente %s terminó de comer con hambre %.2f' % (name, hunger))

In [30]:
env = simpy.Environment()
for i in range(5):
  env.process(attendee(env, i))
env.run(until=220)

El asistente 0 terminó las charlas con un conocimiento 0.17 y hambre 8.00.
El asistente 1 terminó las charlas con un conocimiento 2.60 y hambre 5.00.
El asistente 2 terminó las charlas con un conocimiento 0.50 y hambre 6.00.
El asistente 3 terminó las charlas con un conocimiento 3.85 y hambre 11.00.
El asistente 4 terminó las charlas con un conocimiento 2.50 y hambre 6.00.
El asistente 0 terminó de comer con hambre 0.00
El asistente 1 terminó de comer con hambre 0.00
El asistente 2 terminó de comer con hambre 0.00
El asistente 3 terminó de comer con hambre 4.00
El asistente 4 terminó de comer con hambre 3.00
El asistente 0 terminó las charlas con un conocimiento 0.67 y hambre 5.00.
El asistente 1 terminó las charlas con un conocimiento 7.43 y hambre 6.00.
El asistente 2 terminó las charlas con un conocimiento 3.43 y hambre 9.00.
El asistente 3 terminó las charlas con un conocimiento 3.98 y hambre 9.00.
El asistente 4 terminó las charlas con un conocimiento 2.83 y hambre 9.00.
El asiste

# **Ejemplo: Asistente a la conferencia en el buffet**

In [31]:
TALKS_PER_SESSION = 4
TALK_LENGTH = 30
BREAK_LENGTH = 5
DURATION_EAT = 2
BUFFET_SLOTS = 3

In [32]:
def attendee(env, name, buffet, knowledge=0, hunger=0):
  while True:
    for i in range(TALKS_PER_SESSION):
      knowledge = knowledge + random.randint(0, 3) / (1 + hunger)
      hunger = hunger + random.randint(1, 4)
      yield env.timeout(TALK_LENGTH)
    print('El asistente %s terminó las charlas con un conocimiento %.2f y hambre '
    '%.2f.' % (name, knowledge, hunger))

    start = env.now
    with buffet.request() as req:
      yield req | env.timeout(BREAK_LENGTH - DURATION_EAT)
      time_left = BREAK_LENGTH - (env.now - start)

      if req.triggered:
        food = min(random.randint(3, 12), time_left)
        yield env.timeout(DURATION_EAT)
        hunger -= min(food, hunger)
        time_left -= DURATION_EAT
        print('El asistente %s terminó de comer con hambre %.2f' %
              (name, hunger))
      else:
        hunger += 1
        print('El asistente %s no llegó al buffet :( , el hambre es ahora '
        'at %.2f.' % (name, hunger))
    yield env.timeout(time_left)

In [33]:
env = simpy.Environment()
buffet = simpy.Resource(env, capacity=BUFFET_SLOTS)
for i in range(7):
  env.process(attendee(env, i, buffet))
env.run(until=320)

El asistente 0 terminó las charlas con un conocimiento 2.90 y hambre 14.00.
El asistente 1 terminó las charlas con un conocimiento 4.45 y hambre 5.00.
El asistente 2 terminó las charlas con un conocimiento 4.25 y hambre 7.00.
El asistente 3 terminó las charlas con un conocimiento 4.15 y hambre 7.00.
El asistente 4 terminó las charlas con un conocimiento 1.40 y hambre 9.00.
El asistente 5 terminó las charlas con un conocimiento 1.73 y hambre 9.00.
El asistente 6 terminó las charlas con un conocimiento 1.97 y hambre 12.00.
El asistente 0 terminó de comer con hambre 9.00
El asistente 1 terminó de comer con hambre 0.00
El asistente 2 terminó de comer con hambre 2.00
El asistente 6 no llegó al buffet :( , el hambre es ahora at 13.00.
El asistente 3 terminó de comer con hambre 4.00
El asistente 4 terminó de comer con hambre 6.00
El asistente 5 terminó de comer con hambre 6.00
El asistente 0 terminó las charlas con un conocimiento 3.48 y hambre 15.00.
El asistente 1 terminó las charlas con un

### Asistente a la conferencia en el buffet (configuración de la presentación)

Misma función `attendee` de la celda anterior con los parámetros que usa Stefan Scherfke en las diapositivas: 3 charlas de 30 min, descanso de 15 min, 3 min para comer, 1 puesto en el buffet y 5 asistentes.

In [34]:
TALKS_PER_SESSION = 3
TALK_LENGTH = 30
BREAK_LENGTH = 15
DURATION_EAT = 3
BUFFET_SLOTS = 1

env = simpy.Environment()
buffet = simpy.Resource(env, capacity=BUFFET_SLOTS)
for i in range(5):
  env.process(attendee(env, i, buffet))
env.run(until=220)

El asistente 0 terminó las charlas con un conocimiento 0.58 y hambre 8.00.
El asistente 1 terminó las charlas con un conocimiento 2.17 y hambre 9.00.
El asistente 2 terminó las charlas con un conocimiento 0.83 y hambre 3.00.
El asistente 3 terminó las charlas con un conocimiento 5.50 y hambre 6.00.
El asistente 4 terminó las charlas con un conocimiento 2.75 y hambre 7.00.
El asistente 0 terminó de comer con hambre 0.00
El asistente 1 terminó de comer con hambre 3.00
El asistente 2 terminó de comer con hambre 0.00
El asistente 3 terminó de comer con hambre 0.00
El asistente 4 no llegó al buffet :( , el hambre es ahora at 8.00.
El asistente 0 terminó las charlas con un conocimiento 3.23 y hambre 11.00.
El asistente 1 terminó las charlas con un conocimiento 3.00 y hambre 8.00.
El asistente 2 terminó las charlas con un conocimiento 1.83 y hambre 6.00.
El asistente 3 terminó las charlas con un conocimiento 6.50 y hambre 8.00.
El asistente 4 terminó las charlas con un conocimiento 3.43 y ham

## 5.4.2. Tutorial SimPy in 10 minutes

https://simpy.readthedocs.io/en/latest/simpy_intro/index.html

### Basic Concepts

Un auto que alterna entre estacionar y conducir. El proceso es una función generadora que cede eventos `timeout`.

In [35]:
def car(env):
    while True:
        print('Start parking at %d' % env.now)
        parking_duration = 5
        yield env.timeout(parking_duration)

        print('Start driving at %d' % env.now)
        trip_duration = 2
        yield env.timeout(trip_duration)

#### Our First Process

In [36]:
env = simpy.Environment()
env.process(car(env))
env.run(until=15)

Start parking at 0
Start driving at 5
Start parking at 7
Start driving at 12
Start parking at 14


### Process Interaction

#### Waiting for a Process

El auto eléctrico espera a que termine el proceso de carga antes de volver a conducir.

In [37]:
class Car(object):
    def __init__(self, env):
        self.env = env
        self.action = env.process(self.run())

    def run(self):
        while True:
            print('Start parking and charging at %d' % self.env.now)
            charge_duration = 5
            yield self.env.process(self.charge(charge_duration))

            print('Start driving at %d' % self.env.now)
            trip_duration = 2
            yield self.env.timeout(trip_duration)

    def charge(self, duration):
        yield self.env.timeout(duration)

In [38]:
env = simpy.Environment()
car = Car(env)
env.run(until=15)

Start parking and charging at 0
Start driving at 5
Start parking and charging at 7
Start driving at 12
Start parking and charging at 14


#### Interrupting Another Process

El conductor interrumpe la carga del auto en t = 3.

In [39]:
def driver(env, car):
    yield env.timeout(3)
    car.action.interrupt()

In [40]:
class Car(object):
    def __init__(self, env):
        self.env = env
        self.action = env.process(self.run())

    def run(self):
        while True:
            print('Start parking and charging at %d' % self.env.now)
            charge_duration = 5
            try:
                yield self.env.process(self.charge(charge_duration))
            except simpy.Interrupt:
                print('Was interrupted. Hope, the battery is full enough ...')

            print('Start driving at %d' % self.env.now)
            trip_duration = 2
            yield self.env.timeout(trip_duration)

    def charge(self, duration):
        yield self.env.timeout(duration)

In [41]:
env = simpy.Environment()
car = Car(env)
env.process(driver(env, car))
env.run(until=15)

Start parking and charging at 0
Was interrupted. Hope, the battery is full enough ...
Start driving at 3
Start parking and charging at 5
Start driving at 10
Start parking and charging at 12


### Shared Resources

#### Basic Resource Usage

Estación de carga de baterías (BCS) con dos puestos, modelada como `Resource`.

In [42]:
def car(env, name, bcs, driving_time, charge_duration):
    yield env.timeout(driving_time)

    print('%s arriving at %d' % (name, env.now))
    with bcs.request() as req:
        yield req

        print('%s starting to charge at %s' % (name, env.now))
        yield env.timeout(charge_duration)
        print('%s leaving the bcs at %s' % (name, env.now))

In [43]:
env = simpy.Environment()
bcs = simpy.Resource(env, capacity=2)

In [44]:
for i in range(4):
    env.process(car(env, 'Car %d' % i, bcs, i*2, 5))

In [45]:
env.run()

Car 0 arriving at 0
Car 0 starting to charge at 0
Car 1 arriving at 2
Car 1 starting to charge at 2
Car 2 arriving at 4
Car 0 leaving the bcs at 5
Car 2 starting to charge at 5
Car 3 arriving at 6
Car 1 leaving the bcs at 7
Car 3 starting to charge at 7
Car 2 leaving the bcs at 10
Car 3 leaving the bcs at 12


## 5.4.3. Ejemplos de SimPy

https://simpy.readthedocs.io/en/latest/examples/index.html#all-examples

### 5.4.3.1. Bank Renege

https://simpy.readthedocs.io/en/latest/examples/bank_renege.html

Cubre `Resource` y eventos de condición. Un mostrador bancario con tiempo de servicio aleatorio y clientes que abandonan la cola (*renege*) cuando se les acaba la paciencia.

In [46]:
RANDOM_SEED = 42
NEW_CUSTOMERS = 5
INTERVAL_CUSTOMERS = 10.0
MIN_PATIENCE = 1
MAX_PATIENCE = 3

def source(env, number, interval, counter):
    for i in range(number):
        c = customer(env, f'Customer{i:02d}', counter, time_in_bank=12.0)
        env.process(c)
        t = random.expovariate(1.0 / interval)
        yield env.timeout(t)

def customer(env, name, counter, time_in_bank):
    arrive = env.now
    print(f'{arrive:7.4f} {name}: Here I am')

    with counter.request() as req:
        patience = random.uniform(MIN_PATIENCE, MAX_PATIENCE)
        results = yield req | env.timeout(patience)

        wait = env.now - arrive

        if req in results:
            print(f'{env.now:7.4f} {name}: Waited {wait:6.3f}')

            tib = random.expovariate(1.0 / time_in_bank)
            yield env.timeout(tib)
            print(f'{env.now:7.4f} {name}: Finished')

        else:
            print(f'{env.now:7.4f} {name}: RENEGED after {wait:6.3f}')

print('Bank renege')
random.seed(RANDOM_SEED)
env = simpy.Environment()

counter = simpy.Resource(env, capacity=1)
env.process(source(env, NEW_CUSTOMERS, INTERVAL_CUSTOMERS, counter))
env.run()

Bank renege
 0.0000 Customer00: Here I am
 0.0000 Customer00: Waited  0.000
 3.8595 Customer00: Finished
10.2006 Customer01: Here I am
10.2006 Customer01: Waited  0.000
12.7265 Customer02: Here I am
13.9003 Customer02: RENEGED after  1.174
23.7507 Customer01: Finished
34.9993 Customer03: Here I am
34.9993 Customer03: Waited  0.000
37.9599 Customer03: Finished
40.4798 Customer04: Here I am
40.4798 Customer04: Waited  0.000
43.1401 Customer04: Finished


### 5.4.3.2. Carwash

https://simpy.readthedocs.io/en/latest/examples/carwash.html

Cubre `Resource` y la espera de otros procesos. Un autolavado con un número limitado de máquinas y autos que llegan a lavarse.

In [47]:
RANDOM_SEED = 42
NUM_MACHINES = 2
WASHTIME = 5
T_INTER = 7
SIM_TIME = 20

class Carwash:

    def __init__(self, env, num_machines, washtime):
        self.env = env
        self.machine = simpy.Resource(env, num_machines)
        self.washtime = washtime

    def wash(self, car):
        yield self.env.timeout(self.washtime)
        pct_dirt = random.randint(50, 99)
        print(f"Carwash removed {pct_dirt}% of {car}'s dirt.")

def car(env, name, cw):
    print(f'{name} arrives at the carwash at {env.now:.2f}.')
    with cw.machine.request() as request:
        yield request

        print(f'{name} enters the carwash at {env.now:.2f}.')
        yield env.process(cw.wash(name))

        print(f'{name} leaves the carwash at {env.now:.2f}.')

def setup(env, num_machines, washtime, t_inter):
    carwash = Carwash(env, num_machines, washtime)

    car_count = itertools.count()

    for _ in range(4):
        env.process(car(env, f'Car {next(car_count)}', carwash))

    while True:
        yield env.timeout(random.randint(t_inter - 2, t_inter + 2))
        env.process(car(env, f'Car {next(car_count)}', carwash))

print('Carwash')
print('Check out http://youtu.be/fXXmeP9TvBg while simulating ... ;-)')
random.seed(RANDOM_SEED)

env = simpy.Environment()
env.process(setup(env, NUM_MACHINES, WASHTIME, T_INTER))

env.run(until=SIM_TIME)

Carwash
Check out http://youtu.be/fXXmeP9TvBg while simulating ... ;-)
Car 0 arrives at the carwash at 0.00.
Car 1 arrives at the carwash at 0.00.
Car 2 arrives at the carwash at 0.00.
Car 3 arrives at the carwash at 0.00.
Car 0 enters the carwash at 0.00.
Car 1 enters the carwash at 0.00.
Car 4 arrives at the carwash at 5.00.
Carwash removed 97% of Car 0's dirt.
Carwash removed 67% of Car 1's dirt.
Car 0 leaves the carwash at 5.00.
Car 1 leaves the carwash at 5.00.
Car 2 enters the carwash at 5.00.
Car 3 enters the carwash at 5.00.
Car 5 arrives at the carwash at 10.00.
Carwash removed 64% of Car 2's dirt.
Carwash removed 58% of Car 3's dirt.
Car 2 leaves the carwash at 10.00.
Car 3 leaves the carwash at 10.00.
Car 4 enters the carwash at 10.00.
Car 5 enters the carwash at 10.00.
Carwash removed 97% of Car 4's dirt.
Carwash removed 56% of Car 5's dirt.
Car 4 leaves the carwash at 15.00.
Car 5 leaves the carwash at 15.00.
Car 6 arrives at the carwash at 16.00.
Car 6 enters the carwash 

### 5.4.3.3. Machine Shop

https://simpy.readthedocs.io/en/latest/examples/machine_shop.html

Cubre interrupciones y `PreemptiveResource`. Un taller con n máquinas que se dañan periódicamente y un reparador que deja sus otras tareas para repararlas.

In [48]:
RANDOM_SEED = 42
PT_MEAN = 10.0
PT_SIGMA = 2.0
MTTF = 300.0
BREAK_MEAN = 1 / MTTF
REPAIR_TIME = 30.0
JOB_DURATION = 30.0
NUM_MACHINES = 10
WEEKS = 4
SIM_TIME = WEEKS * 7 * 24 * 60

def time_per_part():
    t = random.normalvariate(PT_MEAN, PT_SIGMA)
    while t <= 0:
        t = random.normalvariate(PT_MEAN, PT_SIGMA)
    return t

def time_to_failure():
    return random.expovariate(BREAK_MEAN)

class Machine:

    def __init__(self, env, name, repairman):
        self.env = env
        self.name = name
        self.parts_made = 0
        self.broken = False

        self.process = env.process(self.working(repairman))
        env.process(self.break_machine())

    def working(self, repairman):
        while True:
            done_in = time_per_part()
            while done_in:
                start = self.env.now
                try:
                    yield self.env.timeout(done_in)
                    done_in = 0

                except simpy.Interrupt:
                    self.broken = True
                    done_in -= self.env.now - start

                    with repairman.request(priority=1) as req:
                        yield req
                        yield self.env.timeout(REPAIR_TIME)

                    self.broken = False

            self.parts_made += 1

    def break_machine(self):
        while True:
            yield self.env.timeout(time_to_failure())
            if not self.broken:
                self.process.interrupt()

def other_jobs(env, repairman):
    while True:
        done_in = JOB_DURATION
        while done_in:
            with repairman.request(priority=2) as req:
                yield req
                start = env.now
                try:
                    yield env.timeout(done_in)
                    done_in = 0
                except simpy.Interrupt:
                    done_in -= env.now - start

print('Machine shop')
random.seed(RANDOM_SEED)

env = simpy.Environment()
repairman = simpy.PreemptiveResource(env, capacity=1)
machines = [Machine(env, f'Machine {i}', repairman) for i in range(NUM_MACHINES)]
env.process(other_jobs(env, repairman))

env.run(until=SIM_TIME)

print(f'Machine shop results after {WEEKS} weeks')
for machine in machines:
    print(f'{machine.name} made {machine.parts_made} parts.')

Machine shop


Machine shop results after 4 weeks
Machine 0 made 3251 parts.
Machine 1 made 3273 parts.
Machine 2 made 3242 parts.
Machine 3 made 3343 parts.
Machine 4 made 3387 parts.
Machine 5 made 3244 parts.
Machine 6 made 3269 parts.
Machine 7 made 3185 parts.
Machine 8 made 3302 parts.
Machine 9 made 3279 parts.


### 5.4.3.4. Movie Renege

https://simpy.readthedocs.io/en/latest/examples/movie_renege.html

Cubre `Resource`, eventos de condición y eventos compartidos. Una taquilla de cine con tres películas, cuando una se agota todos los que esperan por ella abandonan la cola.

In [49]:
RANDOM_SEED = 42
TICKETS = 50
SELLOUT_THRESHOLD = 2
SIM_TIME = 120

def moviegoer(env, movie, num_tickets, theater):
    with theater.counter.request() as my_turn:
        result = yield my_turn | theater.sold_out[movie]

        if my_turn not in result:
            theater.num_renegers[movie] += 1
            return

        if theater.available[movie] < num_tickets:
            yield env.timeout(0.5)
            return

        theater.available[movie] -= num_tickets
        if theater.available[movie] < SELLOUT_THRESHOLD:
            theater.sold_out[movie].succeed()
            theater.when_sold_out[movie] = env.now
            theater.available[movie] = 0
        yield env.timeout(1)

def customer_arrivals(env, theater):
    while True:
        yield env.timeout(random.expovariate(1 / 0.5))

        movie = random.choice(theater.movies)
        num_tickets = random.randint(1, 6)
        if theater.available[movie]:
            env.process(moviegoer(env, movie, num_tickets, theater))

class Theater(NamedTuple):
    counter: simpy.Resource
    movies: List[str]
    available: Dict[str, int]
    sold_out: Dict[str, simpy.Event]
    when_sold_out: Dict[str, Optional[float]]
    num_renegers: Dict[str, int]

print('Movie renege')
random.seed(RANDOM_SEED)
env = simpy.Environment()

movies = ['Python Unchained', 'Kill Process', 'Pulp Implementation']
theater = Theater(
    counter=simpy.Resource(env, capacity=1),
    movies=movies,
    available=dict.fromkeys(movies, TICKETS),
    sold_out={movie: env.event() for movie in movies},
    when_sold_out=dict.fromkeys(movies),
    num_renegers=dict.fromkeys(movies, 0),
)

env.process(customer_arrivals(env, theater))
env.run(until=SIM_TIME)

for movie in movies:
    if theater.sold_out[movie]:
        sellout_time = theater.when_sold_out[movie]
        num_renegers = theater.num_renegers[movie]
        print(
            f'Movie "{movie}" sold out {sellout_time:.1f} minutes '
            f'after ticket counter opening.'
        )
        print(f'  Number of people leaving queue when film sold out: {num_renegers}')

Movie renege
Movie "Python Unchained" sold out 38.0 minutes after ticket counter opening.
  Number of people leaving queue when film sold out: 16
Movie "Kill Process" sold out 43.0 minutes after ticket counter opening.
  Number of people leaving queue when film sold out: 5
Movie "Pulp Implementation" sold out 28.0 minutes after ticket counter opening.
  Number of people leaving queue when film sold out: 5


### 5.4.3.5. Gas Station Refueling

https://simpy.readthedocs.io/en/latest/examples/gas_station_refuel.html

Cubre `Resource`, `Container` y la espera de otros procesos. Una estación de gasolina con surtidores limitados que comparten un tanque, el cual se reabastece con un camión cisterna.

In [50]:
RANDOM_SEED = 42
STATION_TANK_SIZE = 200
THRESHOLD = 25
CAR_TANK_SIZE = 50
CAR_TANK_LEVEL = [5, 25]
REFUELING_SPEED = 2
TANK_TRUCK_TIME = 300
T_INTER = [30, 300]
SIM_TIME = 1000

def car(name, env, gas_station, station_tank):
    car_tank_level = random.randint(*CAR_TANK_LEVEL)
    print(f'{env.now:6.1f} s: {name} arrived at gas station')
    with gas_station.request() as req:
        yield req

        fuel_required = CAR_TANK_SIZE - car_tank_level
        yield station_tank.get(fuel_required)

        yield env.timeout(fuel_required / REFUELING_SPEED)

        print(f'{env.now:6.1f} s: {name} refueled with {fuel_required:.1f}L')

def gas_station_control(env, station_tank):
    while True:
        if station_tank.level / station_tank.capacity * 100 < THRESHOLD:
            print(f'{env.now:6.1f} s: Calling tank truck')
            yield env.process(tank_truck(env, station_tank))

        yield env.timeout(10)

def tank_truck(env, station_tank):
    yield env.timeout(TANK_TRUCK_TIME)
    amount = station_tank.capacity - station_tank.level
    station_tank.put(amount)
    print(
        f'{env.now:6.1f} s: Tank truck arrived and refuelled station with {amount:.1f}L'
    )

def car_generator(env, gas_station, station_tank):
    for i in itertools.count():
        yield env.timeout(random.randint(*T_INTER))
        env.process(car(f'Car {i}', env, gas_station, station_tank))

print('Gas Station refuelling')
random.seed(RANDOM_SEED)

env = simpy.Environment()
gas_station = simpy.Resource(env, 2)
station_tank = simpy.Container(env, STATION_TANK_SIZE, init=STATION_TANK_SIZE)
env.process(gas_station_control(env, station_tank))
env.process(car_generator(env, gas_station, station_tank))

env.run(until=SIM_TIME)

Gas Station refuelling
  87.0 s: Car 0 arrived at gas station
 105.5 s: Car 0 refueled with 37.0L
 129.0 s: Car 1 arrived at gas station
 148.0 s: Car 1 refueled with 38.0L
 284.0 s: Car 2 arrived at gas station
 305.0 s: Car 2 refueled with 42.0L
 385.0 s: Car 3 arrived at gas station
 398.5 s: Car 3 refueled with 27.0L
 459.0 s: Car 4 arrived at gas station
 460.0 s: Calling tank truck
 481.0 s: Car 4 refueled with 44.0L
 705.0 s: Car 5 arrived at gas station
 750.0 s: Car 6 arrived at gas station
 760.0 s: Tank truck arrived and refuelled station with 188.0L
 779.0 s: Car 6 refueled with 38.0L
 781.5 s: Car 5 refueled with 43.0L
 891.0 s: Car 7 arrived at gas station
 904.0 s: Car 7 refueled with 26.0L


### 5.4.3.6. Process Communication

https://simpy.readthedocs.io/en/latest/examples/process_communication.html

Cubre `Store`. Comunicación entre procesos uno a uno y muchos a uno con `Store`, y uno a muchos con la clase `BroadcastPipe`.

In [51]:
RANDOM_SEED = 42
SIM_TIME = 100

class BroadcastPipe:

    def __init__(self, env, capacity=simpy.core.Infinity):
        self.env = env
        self.capacity = capacity
        self.pipes = []

    def put(self, value):
        if not self.pipes:
            raise RuntimeError('There are no output pipes.')
        events = [store.put(value) for store in self.pipes]
        return self.env.all_of(events)

    def get_output_conn(self):
        pipe = simpy.Store(self.env, capacity=self.capacity)
        self.pipes.append(pipe)
        return pipe

def message_generator(name, env, out_pipe):
    while True:
        yield env.timeout(random.randint(6, 10))

        msg = (env.now, f'{name} says hello at {env.now}')
        out_pipe.put(msg)

def message_consumer(name, env, in_pipe):
    while True:
        msg = yield in_pipe.get()

        if msg[0] < env.now:
            print(
                f'LATE Getting Message: at time {env.now}: '
                f'{name} received message: {msg[1]}'
            )

        else:
            print(f'at time {env.now}: {name} received message: {msg[1]}.')

        yield env.timeout(random.randint(4, 8))

print('Process communication')
random.seed(RANDOM_SEED)
env = simpy.Environment()

pipe = simpy.Store(env)
env.process(message_generator('Generator A', env, pipe))
env.process(message_consumer('Consumer A', env, pipe))

print('\nOne-to-one pipe communication\n')
env.run(until=SIM_TIME)

env = simpy.Environment()
bc_pipe = BroadcastPipe(env)

env.process(message_generator('Generator A', env, bc_pipe))
env.process(message_consumer('Consumer A', env, bc_pipe.get_output_conn()))
env.process(message_consumer('Consumer B', env, bc_pipe.get_output_conn()))

print('\nOne-to-many pipe communication\n')
env.run(until=SIM_TIME)

Process communication

One-to-one pipe communication

at time 6: Consumer A received message: Generator A says hello at 6.
at time 12: Consumer A received message: Generator A says hello at 12.
at time 19: Consumer A received message: Generator A says hello at 19.
at time 26: Consumer A received message: Generator A says hello at 26.
at time 36: Consumer A received message: Generator A says hello at 36.
at time 46: Consumer A received message: Generator A says hello at 46.
at time 52: Consumer A received message: Generator A says hello at 52.
at time 58: Consumer A received message: Generator A says hello at 58.
LATE Getting Message: at time 66: Consumer A received message: Generator A says hello at 65
at time 75: Consumer A received message: Generator A says hello at 75.
at time 85: Consumer A received message: Generator A says hello at 85.
at time 95: Consumer A received message: Generator A says hello at 95.

One-to-many pipe communication

at time 10: Consumer A received message: G

### 5.4.3.7. Event Latency

https://simpy.readthedocs.io/en/latest/examples/latency.html

Cubre `Store`. Separa el retardo de propagación de los eventos (un cable) de los procesos que envían y reciben los mensajes.

In [52]:
SIM_DURATION = 100

class Cable:

    def __init__(self, env, delay):
        self.env = env
        self.delay = delay
        self.store = simpy.Store(env)

    def latency(self, value):
        yield self.env.timeout(self.delay)
        self.store.put(value)

    def put(self, value):
        self.env.process(self.latency(value))

    def get(self):
        return self.store.get()

def sender(env, cable):
    while True:
        yield env.timeout(5)
        cable.put(f'Sender sent this at {env.now}')

def receiver(env, cable):
    while True:
        msg = yield cable.get()
        print(f'Received this at {env.now} while {msg}')

print('Event Latency')
env = simpy.Environment()

cable = Cable(env, 10)
env.process(sender(env, cable))
env.process(receiver(env, cable))

env.run(until=SIM_DURATION)

Event Latency
Received this at 15 while Sender sent this at 5
Received this at 20 while Sender sent this at 10
Received this at 25 while Sender sent this at 15
Received this at 30 while Sender sent this at 20
Received this at 35 while Sender sent this at 25
Received this at 40 while Sender sent this at 30
Received this at 45 while Sender sent this at 35
Received this at 50 while Sender sent this at 40
Received this at 55 while Sender sent this at 45
Received this at 60 while Sender sent this at 50
Received this at 65 while Sender sent this at 55
Received this at 70 while Sender sent this at 60
Received this at 75 while Sender sent this at 65
Received this at 80 while Sender sent this at 70
Received this at 85 while Sender sent this at 75
Received this at 90 while Sender sent this at 80
Received this at 95 while Sender sent this at 85
